# VADMamba: Exploring State Space Models for  Fast Video Anomaly Detection (2025)

## Abstract

**VADMamba** is a **Video Anomaly Detection (VAD)** method that replaces traditional temporal models (CNNs or Transformers) with **Mamba (State Space Models)** to efficiently model long-term temporal dependencies.

VADMamba is based on **multi-task** learning for **frame prediction** and **optical flow reconstruction**, with the proposed **VQ-Mamba Unet (VA-MaU)** framework which incorporates a **Vector Quantization (VQ)** layer and **Mamba-based Non-negative Visual State Space (NVSS)** block.

Unlike **Weakly Supervised Video Anomaly Detection (WSVAD)** methods, VADMamba does **not** use weak labels. Instead, it is trained **only on normal videos** and detects anomalies through **prediction and reconstruction errors**.

**Two** individual **VQ-MaU** networks separately **predict frames and reconstruct** corresponding optical flows, further **boosting accuracy** through a **clip-level fusion** evaluation strategy. Experimental results validate the efficacy of the proposed VADMamba across three benchmark datasets, demonstrating superior performance in inference speed compared to previous work. 

Code is available at https://github.com/jLooo/VADMamba.

---

## Contributions

- **VADMamba:** First work to explore and  apply Mamba to solve VAD, by introducing frame prediction and optical flow reconstruction to achieve highprecision hybrid detection.
- **VQ-MaU:** Enables the retrieval  and discrimination of feature representations from compressed normal features via the VQ layer, and construct NVSS blocks to enhance feature aggregation speed.
- **Clip-level** fusion evaluation strategy: Effectively exploits the advantages of different input features, thereby improving detection performance.

---

## Architecture

| Pipeline | Details |
|-|-|
| <img src='images/VADMambaA.png' width='300'> | **(1) Frame Prediction (FP):** The model takes $t$ consecutive input frames, $I_{1:t}$, to predict the next frame, $I_{t+1}$.<br><br>**(2) Flow Reconstruction (FR):** Based on the predicted frame, $I_{t+1}$, the optical flow $O_t$ is reconstructed.<br><br>**(3) Inference stage:** The anomaly scores $S_p$ and $S_r$, obtained from FP and FR, are combined through a clip-level fusion evaluation strategy to produce a hybrid score $S$. |

As we can intuit, FP is trained first, followed by FR, which is trained using the best-performing FP model. ***Parallel training was avoided*** because the two tasks converge at different rates, making simultaneous optimization difficult and resulting in suboptimal hybrid detection performance.

Each branch is based on a **VQ-MaU (Vector Quantized Mamba U-Net)**:

| Pipeline | Details |
|-|-|
| <img src='images/VADMambaB.png' width='600'> | VQ-MaU designs a symmetric **Unet structure** with the skip connection. <br><br>In the encoder stage:<br><br>**(1) Patch Embedding layer** will segment the input image $x \in \mathbb{R}^{H \times W \times C}$ into a non-overlapping patch of size $4 \times 4$, and map the image to 64 channels to generate the embedded image $x' \in \mathbb{R}^{H/4 \times W/4 \times 64}$. <br><br> **(2) NVSS blocks** are used for each encoder stage, input $x'$ to first NVSS, and perform **Patch Merging** down-sampling [1] (except for the last encoder stage), and finally the encoder feature $ze$ is input to VQ. <br><br>In the decoder stage: <br><br> **(1) The representation features $z^q$** by the VQ compression are input to the decoder NVSS block. <br><br> **(2) The features are coupled** with skip feature stage by stage, again using $D_n$ NVSS blocks per decoder stage, and **Patch Expanding** up-sampling [2] before each NVSS block (except for the first decoder stage). <br><br>Finally, the **Final Projection** layer restores the feature $y'$ to $y$ of the same size as the input $x$ to match prediction or reconstruction.|

*Notes:*<br>
1. Depth n of the codec is 4.
2. For the skip connection, we directly use the element-by-element addition operation.
3. U-Net is adopted primarily to support the frame prediction (FP) and flow reconstruction (FR) tasks, which require detailed spatial reconstruction.

Taking a look inside the **Non-negative Vision State Space:**

| Pipeline | Details |
|-|-|
| <img src='images/VADMambaC.png' width='1000'> | The proposed NVSS block consists of two parts:<br><br>**(1) the Vanilla VSS** or VMamba block, which uses depth-wise convolution for high-level feature extraction. The core is the SS2D (2D-Selective-Scan) block which establishes a global receptive field through complementary traversal paths, enabling each pixel to efficiently gather information from all others across multiple directions. <br><br> <img src='images/VADMambaD.png' width='150'> <br><br>**(2) Non-negative Enhanced (NE) module** is proposed to optimize the model for mode-specific feature aggregation and accelerate convergence (linear mapping and activation functions are used to compute gating signals). First, the features are normalized by Layernorm and linear mapping operations, then VADMamba adopts a variant of the pre-activation design proposed in [3], which reorders the conventional ResNet block (Conv→BN→ReLU) into BN→ReLU→Conv. VADMamba further restructures this order as ReLU→Conv→BN, feeding non-negative features into the convolution, as ReLU ensures that the input features are non-negative, which the authors argue helps alleviate gradient vanishing and reduce overfitting.|

To reduce feature dimensions and provide high-quality hidden features, as well as common normal data with small errors and rare abnormal data with large errors, the authors used **vector quantization** in the bottleneck to compress the features, which maximises the retention of essential information without overfittin:




| Pipeline | Details |
|-|-|
| <img src='images/VADMambaE.png' width='1000'> | The **VQ** module defines a codebook (a collection of vectors) $E \in \mathbb{R}^{K \times d}$, where $K$ is the size of the codebook and $d$ is the dimensionality of each codebook vector. Each codebook vector $e_j$ belongs to $\mathbb{R}^d$. <br><br> The vector quantized representation $z_q$ is determined by looking up the codebook vector closest to the input features $z^e(x) \in \mathbb{R}^{h \times w \times  d}$ in terms of the L2 distance:  $$z_q^k = argmin_j ∥z^e(x) − e_j∥_2,$$ where $j \in {1, 2, . . . , K}$ and ej represents the jth item. <br><br> The training objective of **VQ** is defined as follows:  $$\mathcal{L}_{vq} = ∥sg (z^e(x)) − z^q∥_2 + \beta ∥z^e(x) − sg (z^q)∥_2,$$ where $sg(·)$ represents the stop-gradient operator defined as the identity function $sg(x)=x$ during the forward pass, but blocks the flow of gradients during the backward pass, and $\beta$ set to 0.25. |

The intuition behind the loss function is that the first term moves the codebook toward the encoder, while the second term moves the encoder toward the codebook. The stop-gradient operator determines during backpropagation which side should move and which one should remain fixed.

| Term | Where does the gradient flow? | What does it update? | Intuition |
|---|---|---|---|
| $∥sg(z^e(x))-z_q∥_2$ | **Does not flow through $z_e$**, but does flow through $z_q$ | **Codebook** | The codebook moves toward the features produced by the encoder |
| $\beta∥z^e(x)-sg(z^q)∥_2$ | **Flows through $z_e$**, but not through $z_q$ | **Encoder** | The encoder learns to produce features close to the codebook vectors |


---

## Role of Mamba

Mamba replaces attention-based temporal modeling with **State Space Models (SSMs)**.

Its main advantages include:

- Linear computational complexity.
- Lower memory consumption.
- Efficient modeling of long video sequences.

---

## Main Contribution

The primary contribution of VADMamba is **not** a new anomaly detection paradigm, but rather demonstrating that **Mamba can effectively replace CNNs and Transformers for temporal modeling in reconstruction- and prediction-based VAD**, achieving competitive performance with improved computational efficiency.

---

## Limitations

- Does not use weak supervision.
- Does not leverage semantic information from vision-language models (e.g., CLIP).
- Relies on reconstruction and prediction errors, a paradigm that has gradually become less dominant as modern VAD methods increasingly favor discriminative learning approaches.

---

## Relation to WSVAD

Although VADMamba adopts Mamba for temporal modeling, it belongs to a **different research direction** than Weakly Supervised Video Anomaly Detection (WSVAD).

- **VADMamba:** Learns the distribution of normal videos and detects anomalies through reconstruction/prediction errors.
- **WSVAD:** Learns discriminative representations from weak labels (e.g., MIL, CLIP-based methods) to directly localize anomalous segments.

---

## Key Takeaway

> **VADMamba demonstrates that Mamba is an efficient temporal modeling backbone for reconstruction- and prediction-based Video Anomaly Detection, but it does not address the core challenges of Weakly Supervised Video Anomaly Detection.**

## Quantitative Comparison on Video Anomaly Detection Benchmarks

| Method | Model | Ped2 | Avenue | SHT | FPS |
|--------|:-----:|-----:|--------:|----:|----:|
| **Single-Task Methods** ||||| |
| Frame-Pred [11] | CNN | 95.4 | 85.1 | 72.8 | 25 |
| MemAE [7] | CNN | 94.1 | 83.3 | 71.2 | 45 |
| MNAD [8] | CNN | 97.0 | 88.5 | 70.5 | 78 |
| VECI [15] | CNN | 97.3 | 90.2 | 74.8 | – |
| TMAE [16] | ViT | 94.1 | 89.8 | 71.4 | – |
| MSN-net [12] | CNN | 97.6 | 89.4 | 73.4 | 95 |
| STGCN-FFP [13] | CNN | 96.9 | 88.4 | 73.7 | – |
| A2D-GAN [9] | GAN | 97.4 | **91.0** | 74.2 | – |
| VADCL [10] | ViT | 92.2 | 86.2 | 73.8 | – |
| USTN-DSC [5] | ViT | **98.1** | 89.9 | 73.8 | – |
| VADMamba w/o FP | SSM | 96.2 | 87.3 | 72.2 | 132 |
| VADMamba w/o FR | SSM | 97.9 | 86.2 | 74.4 | 151 |
| **Hybrid Methods** ||||| |
| AnoPCN [17] | CNN | 96.8 | 86.2 | 73.6 | 10 |
| STM-AE [18] | CNN | 98.1 | 89.8 | 73.8 | 40 |
| MGAN-CL [19] | GAN | 96.5 | 87.1 | 73.6 | 30 |
| MAAM-Net [20] | CNN | 97.7 | 90.9 | 71.3 | 64 |
| GroupGAN [21] | GAN | 96.6 | 85.5 | 73.1 | 70 |
| C²Net [22] | GAN | 98.0 | 87.5 | 71.4 | – |
| PDM-Net [23] | CNN | 97.7 | 88.1 | 74.2 | – |
| **VADMamba (Ours)** | **SSM** | **98.5** | **91.5** | **77.0** | **90** |

## Why Doesn't VADMamba Compare with Sultani or Other WSVAD Methods?

One important observation is that **VADMamba does not compare against methods such as Sultani et al. (MIL), RTFM, VADCLIP, DMU, or other Weakly Supervised Video Anomaly Detection (WSVAD) approaches.**

This is because **VADMamba belongs to a different research paradigm.**

Instead of solving **Weakly Supervised Video Anomaly Detection**, it focuses on **Unsupervised/Self-Supervised Video Anomaly Detection**, where the model is trained **only with normal videos** and detects anomalies through prediction and reconstruction errors.

Consequently, VADMamba is evaluated against methods from the same research direction, including reconstruction- and prediction-based approaches such as:

- Future Frame Prediction
- MemAE
- MNAD
- HF²-VAD
- MIST
- Other reconstruction/prediction-based methods

It does **not** compare with:

- Sultani et al. (MIL)
- RTFM
- VADCLIP
- DMU
- UMIL
- Other WSVAD methods

because these methods require **video-level weak labels** during training, making the experimental protocols fundamentally different.

---

## Benchmark Datasets

A simple way to distinguish the two research directions is by the datasets they use.

| Unsupervised VAD | Weakly Supervised VAD |
|------------------|-----------------------|
| UCSD Ped2 | UCF-Crime |
| CUHK Avenue | XD-Violence |
| ShanghaiTech Campus | UBnormal (recent works) |

In general:

- Papers evaluated on **Ped2, Avenue, and ShanghaiTech** typically belong to the **unsupervised reconstruction/prediction** family.
- Papers evaluated on **UCF-Crime** and **XD-Violence** usually belong to **Weakly Supervised Video Anomaly Detection (WSVAD)**.

---

## Key Insight

VADMamba is **not intended to outperform WSVAD methods** such as RTFM or VADCLIP.

Instead, its primary contribution is to demonstrate that **Mamba serves as an efficient temporal modeling backbone for reconstruction- and prediction-based Video Anomaly Detection**, offering lower computational complexity while maintaining competitive performance.

Therefore, for researchers working on **Weakly Supervised Video Anomaly Detection**, VADMamba should primarily be viewed as a reference on **how Mamba can be applied for temporal modeling**, rather than as a direct methodological competitor.

### References

[1] Liu, Z., Lin, Y., Cao, Y., Hu, H., Wei, Y., Zhang, Z., ... & Guo, B. (2021, October). Swin transformer: Hierarchical vision transformer using shifted windows. In *2021 IEEE/CVF international conference on computer vision (ICCV)* (pp. 9992-10002). Ieee.

[2] Cao, H., Wang, Y., Chen, J., Jiang, D., Zhang, X., Tian, Q., & Wang, M. (2022, October). Swin-unet: Unet-like pure transformer for medical image segmentation. In *European conference on computer vision (pp. 205-218)*. Cham: Springer Nature Switzerland.

[3] He, K., Zhang, X., Ren, S., & Sun, J. (2016, September). Identity mappings in deep residual networks. In *European conference on computer vision (pp. 630-645)*. Cham: Springer International Publishing.